# 2. Entity-Centric Investigation

In notebook 1 you picked the highest-priority incident. Now you have to **investigate** it.

The big lesson of this notebook: a detection tells you **one fact** (e.g. "15 failed sign-ins for alice"). A full investigation **pivots on entities** — user, IP, device, email — across every data source until the full kill chain is visible.

The seeded attack looks like this:

```
Stage 1  Phishing email delivered to alice@contoso.com
Stage 2  15 failed + 1 successful sign-in from Moscow (brute force)
Stage 3  psexec.exe + mimikatz.exe on laptop-alice → spread to VMs
Stage 4  Large outbound uploads to known-bad IPs (exfiltration)
```

Your job: **prove all four stages** by pivoting on entities.

> **SC-200 mapping**: "Investigate alerts and incidents in Microsoft Defender XDR" and "Hunt threats with KQL".


## Setup

This lab reuses the mini-SIEM from Lab 1. Make sure it's running:

```bash
cd ../../01-build-a-siem && docker compose up -d
```

Then, in VS Code:
1. Pick the **`.venv` kernel** from this folder (top-right kernel picker).
2. If it's missing, reload the window (`Cmd+Shift+P` → `Reload Window`).

All cells talk to `http://localhost:8000` — the same mini-SIEM container seeded with a realistic multi-stage attack plus normal background traffic.


In [1]:
import httpx
from collections import Counter, defaultdict
from datetime import datetime

SIEM = 'http://localhost:8000'

# Sanity check: can we reach the SIEM?
health = httpx.get(f'{SIEM}/health').json()
print('SIEM health:', health)

dashboard = httpx.get(f'{SIEM}/dashboard').json()
print('Dashboard:', dashboard)


SIEM health: {'status': 'ok', 'service': 'mini-siem', 'counts': {'logs': 172, 'analytics_rules': 6, 'alerts': 6, 'incidents': 6, 'playbooks': 5}}
Dashboard: {'total_logs': 172, 'tables': ['AzureFirewall', 'DeviceEvents', 'EmailEvents', 'SigninLogs'], 'active_rules': 6, 'open_alerts': 0, 'open_incidents': 0, 'severity_breakdown': {'High': 4, 'Medium': 2}}


## ❌ Bad: tunnel vision on the triggering alert

A junior analyst reads the brute-force alert, says "alice's password is weak", forces a password reset, and closes the incident.

They just missed three other stages of the attack. Let's show what they saw:


In [2]:
# Pick the brute-force incident (highest-severity, CredentialAccess tactic)
incidents = httpx.get(f'{SIEM}/incidents').json()
brute = next(
    (i for i in incidents if 'Brute force' in i['title']),
    incidents[0],
)
detail = httpx.get(f"{SIEM}/incidents/{brute['id']}").json()

print(f"Incident: {detail['title']}  ({detail['severity']})")
print(f"Alerts attached: {len(detail['alerts'])}")
for a in detail['alerts']:
    print(f"  • {a['rule_name']} — {a['title']}")
print('\n❌ Stopping here would miss phishing, lateral movement, and exfiltration.')


Incident: Incident: Brute force sign-in (UserPrincipalName=alice@contoso.com)  (High)
Alerts attached: 1
  • Brute force sign-in — Brute force sign-in: alice@contoso.com (15 events)

❌ Stopping here would miss phishing, lateral movement, and exfiltration.


## ✅ Best: pivot on entities, across data sources

From the incident we pull the **primary entity** (the user) and let it drive queries into every relevant table.

> Real Defender XDR does this visually with the **Incident graph**; Sentinel does it with entity pages and KQL joins. We're rebuilding that flow by hand so the idea sticks.


In [3]:
import json as _json
entities = _json.loads(detail['entities'] or '{}')
primary_user = entities.get('UserPrincipalName', 'alice@contoso.com')
print(f'Primary entity: user = {primary_user}')


Primary entity: user = alice@contoso.com


### Stage 1 — Initial Access (email)

**MITRE tactic**: `TA0001 Initial Access`. If the user was phished, we should see an email from a look-alike sender.


In [4]:
r = httpx.post(f'{SIEM}/query', json={
    'table_name': 'EmailEvents',
    'filter': {'RecipientEmailAddress': primary_user},
    'limit': 20,
})
emails = r.json()['results']
phish = [e for e in emails if e.get('ThreatTypes') == 'Phish']

print(f"Emails to {primary_user}: {len(emails)}   Phish flagged: {len(phish)}")
for e in phish:
    mark = '📬 DELIVERED' if e['DeliveryAction'] == 'Delivered' else '🚫 blocked'
    print(f"  {mark}  from {e['SenderFromAddress']}  →  {e['Subject']}")


Emails to alice@contoso.com: 5   Phish flagged: 1
  🚫 blocked  from security-alert@m1crosoft-support.com  →  Urgent: Your account has been compromised - verify now


### Stage 2 — Credential Access (sign-ins)

**MITRE tactic**: `TA0006 Credential Access`. Brute force usually looks like many failures from one or two IPs, possibly from unusual locations.


In [5]:
r = httpx.post(f'{SIEM}/query', json={
    'table_name': 'SigninLogs',
    'filter': {'UserPrincipalName': primary_user},
    'limit': 50,
})
signins = r.json()['results']
fail = [s for s in signins if s['ResultType'] == 'Failure']
succ = [s for s in signins if s['ResultType'] == 'Success']
print(f'Sign-ins: {len(signins)}   failures: {len(fail)}   successes: {len(succ)}')

SUS_LOC = {'Moscow', 'Beijing', 'Anonymous Proxy'}
bad_ips = Counter(s['IPAddress'] for s in fail)
print('Top failure IPs:', bad_ips.most_common(3))
print('Locations:')
for loc, n in Counter(s['Location'] for s in signins).most_common():
    mark = ' ⚠️' if loc in SUS_LOC else ''
    print(f'  {loc}: {n}{mark}')

# Pivot: remember the suspicious IP — we'll use it in stage 4
attacker_ip = bad_ips.most_common(1)[0][0] if bad_ips else None
print(f'\nPivot target → attacker IP: {attacker_ip}')


Sign-ins: 21   failures: 15   successes: 6
Top failure IPs: [('185.220.101.42', 15)]
Locations:
  Moscow: 7 ⚠️
  Beijing: 5 ⚠️
  Anonymous Proxy: 4 ⚠️
  New York: 2
  Office: 2
  Seattle: 1

Pivot target → attacker IP: 185.220.101.42


### Stage 3 — Execution & Lateral Movement (endpoint)

**MITRE tactics**: `TA0002 Execution` + `TA0008 Lateral Movement`. After cred theft, attackers run tooling like `psexec`, `mimikatz`, `certutil`, `cmd`, or `powershell`.


In [6]:
username = primary_user.split('@')[0]
r = httpx.post(f'{SIEM}/query', json={
    'table_name': 'DeviceEvents',
    'filter': {'AccountName': username},
    'limit': 50,
})
endpoint = r.json()['results']

SUS_TOOLS = {'mimikatz.exe', 'psexec.exe', 'certutil.exe'}
sus = [e for e in endpoint if e['FileName'] in SUS_TOOLS]
print(f'Endpoint events for {username}: {len(endpoint)}   known-bad tooling: {len(sus)}')

for e in sus:
    print(f"  🔴 {e['DeviceName']:<14}  {e['FileName']:<14} {e['ActionType']}   path={e['FolderPath']}")

print('\nDevices touched:')
for dev, n in Counter(e['DeviceName'] for e in endpoint).most_common():
    print(f'  {dev}: {n} events')


Endpoint events for alice: 26   known-bad tooling: 6
  🔴 vm-web-01       mimikatz.exe   ProcessCreated   path=C:\Users\Downloads\
  🔴 laptop-alice    psexec.exe     RemoteExecution   path=C:\Windows\System32\
  🔴 vm-db-01        mimikatz.exe   ProcessCreated   path=C:\Windows\System32\
  🔴 laptop-alice    psexec.exe     RemoteExecution   path=/usr/bin/
  🔴 vm-app-01       mimikatz.exe   ProcessCreated   path=C:\Windows\System32\
  🔴 laptop-alice    psexec.exe     RemoteExecution   path=C:\Windows\System32\

Devices touched:
  vm-app-01: 15 events
  laptop-alice: 4 events
  laptop-bob: 3 events
  vm-web-01: 2 events
  vm-db-01: 2 events


### Stage 4 — Exfiltration (network)

**MITRE tactic**: `TA0010 Exfiltration`. Connections to known-bad destinations are a strong signal. In a real tenant you'd match against a threat-intel feed — here we use a hard-coded list.


In [7]:
KNOWN_BAD_IPS = ['185.220.101.42', '45.33.32.156', '198.51.100.99']
exfil = []
for ip in KNOWN_BAD_IPS:
    r = httpx.post(f'{SIEM}/query', json={
        'table_name': 'AzureFirewall',
        'filter': {'DestinationIP': ip},
        'limit': 20,
    })
    rows = r.json()['results']
    if rows:
        exfil.extend(rows)
        srcs = Counter(row['SourceIP'] for row in rows)
        print(f'⚠️  {len(rows)} connections to {ip}')
        for s, n in srcs.most_common():
            print(f'       from {s}  × {n}')

print(f'\nTotal suspicious outbound flows: {len(exfil)}')


⚠️  4 connections to 185.220.101.42
       from 10.0.2.10  × 4
⚠️  5 connections to 45.33.32.156
       from 10.0.2.10  × 5
⚠️  1 connections to 198.51.100.99
       from 10.0.2.10  × 1

Total suspicious outbound flows: 10


## Write the incident report

The investigation isn't over until you **write it down**. A good incident summary contains: timeline, entities touched, MITRE tactics seen, and recommended response.

Below we generate one automatically from the data we collected, then attach it as a comment on the incident (`PATCH /incidents/{id}`).


In [8]:
report_lines = [
    f"Incident {detail['id']} — {detail['title']}",
    f"User: {primary_user}    Attacker IP: {attacker_ip}",
    f"Stage 1 Initial Access : {len(phish)} phishing email(s) to the user",
    f"Stage 2 Credential Access: {len(fail)} failed sign-ins, {len(succ)} success from suspicious IP",
    f"Stage 3 Execution/LatMov : {len(sus)} known-bad tool executions across {len({e['DeviceName'] for e in sus})} device(s)",
    f"Stage 4 Exfiltration     : {len(exfil)} outbound flows to known-bad IPs",
    "Recommended response:",
    "  1. Contain — isolate laptop-alice, disable alice's account",
    "  2. Eradicate — remove mimikatz/psexec, reset credentials, revoke tokens",
    "  3. Recover — re-image affected hosts, re-enable account with MFA",
]
report = '\n'.join(report_lines)
print(report)

# Persist the report as an incident comment (safe to re-run; PATCH will append)
httpx.patch(
    f"{SIEM}/incidents/{detail['id']}",
    json={'comment': report},
)
print('\n📝 Report attached to incident as a comment.')


Incident INC-ddb928 — Incident: Brute force sign-in (UserPrincipalName=alice@contoso.com)
User: alice@contoso.com    Attacker IP: 185.220.101.42
Stage 1 Initial Access : 1 phishing email(s) to the user
Stage 2 Credential Access: 15 failed sign-ins, 6 success from suspicious IP
Stage 3 Execution/LatMov : 6 known-bad tool executions across 4 device(s)
Stage 4 Exfiltration     : 10 outbound flows to known-bad IPs
Recommended response:
  1. Contain — isolate laptop-alice, disable alice's account
  2. Eradicate — remove mimikatz/psexec, reset credentials, revoke tokens
  3. Recover — re-image affected hosts, re-enable account with MFA

📝 Report attached to incident as a comment.


## What you just did (SC-200 mapping)

| You did... | Real portal equivalent |
|---|---|
| Picked the incident's primary entity | Incident page → Assets tab |
| Pivoted user → email / sign-in / endpoint / network | Defender XDR incident graph |
| Tagged each finding with a MITRE tactic | ATT&CK column on alerts & incidents |
| Wrote a timeline report as a comment | Incident → Comments / Activity log |

### Exam tips

- The investigation is complete only when you can name the **full kill chain** and every **entity** touched.
- A brute-force alert is rarely the whole story. Always pivot the user forward (exec, lateral, exfil) and backward (email, phishing).
- Network evidence (firewall + proxy) is the usual confirmation of exfiltration.

➡️ Next: [03 — Incident lifecycle & classification](03_incident_lifecycle.ipynb)
